<a href="https://colab.research.google.com/github/prometheus404/NLP_proj/blob/master/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Initiazlization

In [ ]:
%pip install llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu122

Looking in indexes: https://pypi.org/simple, https://abetlen.github.io/llama-cpp-python/whl/cu122


In [ ]:
#DRIVE
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from llama_cpp import Llama, llama_free, llama_free_model
from tqdm import tqdm
#from transformers import AutoTokenizer, pipeline, BitsAndBytesConfig
import requests
from collections import defaultdict
import json
import torch
import os

# Params
VERBOSE = True
CHOSEN = 'llama'
FILE_NAMES = ['ticket_to_ride', 'dominion', 'catan', 'power_grid_recharged']
IT = 5
BASE_URL = 'https://raw.githubusercontent.com/prometheus404/NLP_proj/refs/heads/master/rules/flawed_texts/'
BASE_FOLDER = 'drive/MyDrive/NLP_proj/error_detection/'
OVERWRITE = ['']

# Load the model
models = {
    'llama': {'repo_id':"bartowski/Meta-Llama-3.1-8B-Instruct-GGUF",
              'filename':"Meta-Llama-3.1-8B-Instruct-Q8_0.gguf",
              'temperature': 0.7,
              'n_ctx': 32768,
              'chat_format': "llama-3"
              },
    'qwen': {'repo_id':"Qwen/Qwen3-8B-GGUF",
             'filename': "Qwen3-8B-Q8_0.gguf",
             'temperature': 0.6,
             'n_ctx': 40960,
             'chat_format': "qwen"},
    'mistral': {'repo_id':"TheBloke/Mistral-7B-v0.1-GGUF",
                'filename':"mistral-7b-v0.1.Q8_0.gguf",
                'temperature': 0.7,
                'n_ctx': 32768,
                'chat_format': "chatml"},
}

model = Llama.from_pretrained(repo_id=models[CHOSEN]['repo_id'], # repository name
                            filename=models[CHOSEN]['filename'], # model file
                            n_gpu_layers=-1, # use all GPU layers
                            n_ctx=models[CHOSEN]['n_ctx'], # context size
                            flash_attn=True, # use flash attention
                            chat_format=models[CHOSEN]['chat_format'], # chat format
                            verbose=False,
                            force_download=True,
                            enable_thinking=True)

def generate_message(sys_prompt, usr_prompt):
    return [
        {
                "role": "system",
                "content": sys_prompt,
            },
            {
                "role": "user",
                "content": usr_prompt,
            },
    ]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


# Error detection

 Each rulebook is edited by inserting errors that are increasingly difficult to spot:
 - level 0: **original**    -> unaltered rulebook
 - level 1: **missing**     -> an entire paragraph of the rulebook describing some core mechanic is missing
- level 2: **unsolvable**   -> a rule directly contradicts another one
- level 3: **incoherent**   -> a combination of rules hardlocks the game
- level 4: **gamebreaking** -> a coherent but obviously unbalanced mechanic

In [ ]:
prompt = """You are an expert game board player.
Examine the rulebook provided by the user.
Proceed with a chain of thought:

- Scan the text linearly and note any statements that conflict with earlier ones.
- For each of the game mechanic check if it is explained.
- Check whether any mechanic could halt the game or give a player an overwhelming advantage.

Report the **most impactful** problem you discover, quoting the relevant line and summarizing its impact.
If nothing stands out, reply “The rules appear consistent.”
"""



output_dict = {str(lvl): {str(it): '' for it in range(IT)} for lvl in range(5)}
to_do = [g for g in FILE_NAMES if '{CHOSEN}_{g}.json' not in os.listdir(BASE_FOLDER)
                                 or '{CHOSEN}_{g}.json' in OVERWRITE]

if to_do == []:
    print('Nothing to do')

for game in to_do:
    for it,lvl in tqdm([(it,lvl) for it in range(iterations) for lvl in range(5)]):
    rulebook = requests.get(BASE_URL+'lvl'+str(lvl)+'/'+game+'.txt').text #rulebook is cached so should not be a problem
    name = game.replace('_',' ')
    out = model.create_chat_completion(generate_message(prompt, "Here is the full rulebook of the game {name}:\n"+rulebook),
                                       temperature=models[CHOSEN]['temperature'])['choices'][0]['message']['content']
    output_dict['lvl'+str(lvl)][str(it)] = out
    if(VERBOSE):
        print(f'{game} lvl{lvl}, iteration: {it}\n---------\n{out}')


    with open(f'{BASE_DIR}{CHOSEN}_{game}_error.json','w') as f:
        json.dump(dict(output_dict),f)

  1%|          | 1/100 [01:09<1:54:54, 69.64s/it]

ticket_to_ride lvl0, iteration: 0
---------
<think>
Okay, let me go through the rules step by step. The user wants me to check for inconsistencies, unexplained mechanics, or any issues that could halt the game or give an advantage.

First, looking at the setup: each player starts with 45 trains. The game ends when a player has 0, 1, or 2 trains left, then everyone gets a final turn. That seems okay. 

Next, the game turn actions: draw train cards, claim a route, or draw tickets. The draw train cards section says you can draw 2 cards, either from face-up or the deck. But there's a note that if you draw a face-up locomotive, it has to be the first card, and you can't draw a second. Wait, but if you draw a locomotive from the face-up pile first, you can't draw another. However, if you draw a locomotive from the deck (blind draw), you can still draw a second card. That seems a bit inconsistent. Let me check again.

In the "Draw Train Cards" section: "If you want to draw a face up locomotiv

  2%|▏         | 2/100 [02:29<2:03:46, 75.78s/it]

ticket_to_ride lvl1, iteration: 0
---------
<think>
Okay, let's go through the rules step by step. The user wants me to check for inconsistencies or problematic mechanics.

First, looking at the setup: each player starts with 45 trains. The game ends when a player has 0, 1, or 2 trains left. Then everyone gets a final turn. Wait, the problem here might be that if a player runs out of trains, they get a final turn, but the other players might still be able to play. But the rules say "each player, including that player, gets 1 final turn." So even if someone has few trains left, they get a final turn. But does that mean the game ends after that? Let me check the game end section again.

The game ends when a player's stock is down to 0,1, or 2. Then each player gets a final turn. Then they calculate scores. So the game doesn't end immediately when someone hits that threshold; instead, everyone gets one more turn. That seems okay. But maybe there's an issue with the game ending too early o

  3%|▎         | 3/100 [03:20<1:44:02, 64.35s/it]

...

catan lvl0, iteration: 3
---------
<think>
Okay, let's go through the rules step by step. The user wants me to check for inconsistencies, unexplained mechanics, or any mechanic that could halt the game or give an advantage.

First, looking at the setup. The initial setup mentions using the setup shown, but the variable setup is described later. Wait, the variable setup is an alternative, but the main setup is the first one. But maybe that's okay. 

Next, the objective is to reach 10 VP. The VP tiles are Longest Route (2 VP) and Largest Army (2 VP). But the victory point cards are separate, with 5x victory point cards. So players can get VP from buildings (settlements give 1, cities 2), development cards (like the victory point card gives VP when played), and the two tiles. But the rules say that the first player to reach 10 VP on their turn wins. 

Now, checking the Production Phase. When rolling a 7, players must discard half their resources (rounded down). Then activate the robber. T

 67%|██████▋   | 67/100 [2:11:37<1:23:46, 152.32s/it]

catan lvl1, iteration: 3
---------
<think>
Okay, let's go through the rules step by step. The user wants me to check for inconsistencies or problematic mechanics. 

First, looking at the setup section. The initial setup mentions placing 2 starting settlements and roads for each player. Then in the variable setup, round 1 has the first player placing 1 settlement and 1 road, then others do the same. Round 2 has players placing a second settlement and road. Wait, in the standard setup, each player starts with 2 settlements and roads. But in the variable setup, round 1 gives 1 settlement and 1 road, then round 2 adds another settlement and road. That seems okay.

Now, the production phase. When a player rolls a 7, they have to discard half their resources (rounded down) and move the robber. Then steal a resource. But the example says if a player has 9 cards, they discard 4. That's correct because half of 9 is 4.5, rounded down to 4. 

Wait, the problem might be in the variable setup's ste

 68%|██████▊   | 68/100 [2:16:10<1:40:35, 188.61s/it]

catan lvl2, iteration: 3
---------
<think>
Okay, let's go through the rules step by step. The user wants me to check for inconsistencies, unexplained mechanics, or potential game-breaking issues.

First, looking at the setup. The standard setup mentions placing 2 starting settlements and roads for each player. But in the variable setup, Round 1 has the first player placing 1 settlement and 1 road, then others do the same. Round 2 has players placing a second settlement and road. Wait, in the standard setup, each player starts with 2 settlements and roads. But in the variable setup, Round 1 gives 1 settlement and road, then Round 2 adds another settlement and road. That seems to align with the standard setup. Wait, but the standard setup says "Place 2 starting settlements and roads for each player as shown." Maybe the variable setup is different. Wait, the variable setup's Round 1 gives each player 1 settlement and 1 road, then Round 2 adds another settlement and road. So each player en